In [ ]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import process, fuzz

In [ ]:
def fix_encoding(name):
    if not isinstance(name, str): return ""
    try:
        return name.encode('raw_unicode_escape').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        return name

def normalize_name(name):
    if not isinstance(name, str): return ""
    normalized = unicodedata.normalize('NFD', fix_encoding(name))
    ascii_name = normalized.encode('ascii', 'ignore').decode("utf-8").lower().strip()
    ascii_name = re.sub(r'[^a-z0-9 ]', ' ', ascii_name)
    return re.sub(r'\s+', ' ', ascii_name).strip()

In [32]:
# ── 1. Chargement ─────────────────────────────────────────────────────────────
df_mapping = pd.read_csv("../data_finale/mapping_fbref_tm.csv", encoding='latin1')
df_fbref   = pd.read_csv("../data/fbref_datasets/players_data-2025_2026.csv")
df_tm      = pd.read_csv("../data/transfermarkt_datasets/players.csv")

# ── 2. Normalisation ──────────────────────────────────────────────────────────
df_mapping['PlayerFBref'] = df_mapping['PlayerFBref'].apply(fix_encoding)
df_mapping['join_key']    = df_mapping['PlayerFBref'].apply(normalize_name)

df_fbref['join_key'] = df_fbref['Player'].apply(normalize_name)
# Fix 3 : Born dans FBref = année seule (ex: 1999)
df_fbref['dob_key']  = df_fbref['Born'].astype(str).str.strip()

df_tm['join_key']      = (df_tm['first_name'].apply(normalize_name) + ' ' + df_tm['last_name'].apply(normalize_name)).str.strip()
df_tm['join_key_full'] = df_tm['name'].apply(normalize_name)
# Fix 3 : on extrait uniquement l'année depuis date_of_birth TM pour matcher avec Born FBref
df_tm['dob_key']       = pd.to_datetime(df_tm['date_of_birth'], errors='coerce').dt.strftime('%Y')

results   = []
remaining = df_fbref.copy()

def remove_matched(df, keys_matched):
    return df[~df['join_key'].isin(keys_matched)].copy()

# ══════════════════════════════════════════════════════════════════════════════
# NIVEAU 1 — Nom exact via mapping
# ══════════════════════════════════════════════════════════════════════════════
merge_1 = pd.merge(remaining, df_mapping[['join_key', 'tm_id']], on='join_key', how='inner')
merge_1['match_method'] = 'exact_name_mapping'
results.append(merge_1)
remaining = remove_matched(remaining, set(merge_1['join_key']))
print(f"[1] Nom exact (mapping)     : {len(merge_1):>5} | restants : {len(remaining)}")

# ══════════════════════════════════════════════════════════════════════════════
# NIVEAU 2 — Fuzzy nom via mapping
# ══════════════════════════════════════════════════════════════════════════════
SCORE_MIN = 90
mapping_keys = df_mapping['join_key'].tolist()
fuzzy_rows = []

for _, row in remaining.iterrows():
    res = process.extractOne(row['join_key'], mapping_keys, scorer=fuzz.token_sort_ratio)
    if res and res[1] >= SCORE_MIN:
        tm_id = df_mapping[df_mapping['join_key'] == res[0]].iloc[0]['tm_id']
        fuzzy_rows.append({**row.to_dict(), 'tm_id': tm_id, 'match_method': f'fuzzy_name({res[1]})'})

merge_2 = pd.DataFrame(fuzzy_rows)
if not merge_2.empty:
    results.append(merge_2)
    remaining = remove_matched(remaining, set(merge_2['join_key']))
print(f"[2] Fuzzy nom (mapping)     : {len(merge_2):>5} | restants : {len(remaining)}")

# ══════════════════════════════════════════════════════════════════════════════
# NIVEAU 3 — Nom exact + DOB (FBref ↔ TM directement, sans mapping)
# ══════════════════════════════════════════════════════════════════════════════
tm_slim = df_tm[['player_id', 'join_key', 'dob_key']].rename(columns={'player_id': 'tm_id'})
merge_3a = pd.merge(remaining, tm_slim, on=['join_key', 'dob_key'], how='inner')

tm_slim_full = df_tm[['player_id', 'join_key_full', 'dob_key']].rename(
    columns={'player_id': 'tm_id', 'join_key_full': 'join_key'}
)
merge_3b = pd.merge(remaining, tm_slim_full, on=['join_key', 'dob_key'], how='inner')

merge_3 = pd.concat([merge_3a, merge_3b]).drop_duplicates(subset='join_key')
merge_3['match_method'] = 'exact_name+dob'
results.append(merge_3)
remaining = remove_matched(remaining, set(merge_3['join_key']))
print(f"[3] Nom exact + DOB (TM)    : {len(merge_3):>5} | restants : {len(remaining)}")

# ══════════════════════════════════════════════════════════════════════════════
# NIVEAU 4 — DOB seul + fuzzy nom (FBref ↔ TM) pour les cas difficiles
# ══════════════════════════════════════════════════════════════════════════════
tm_dob = df_tm[['player_id', 'join_key', 'join_key_full', 'dob_key']].rename(columns={'player_id': 'tm_id'})
candidates = pd.merge(remaining, tm_dob, on='dob_key', how='inner', suffixes=('_fbref', '_tm'))

fuzzy_dob_rows = []
for _, row in candidates.iterrows():
    score_1 = fuzz.token_sort_ratio(row['join_key_fbref'], row['join_key_tm'])
    score_2 = fuzz.token_sort_ratio(row['join_key_fbref'], row['join_key_full'])
    best_score = max(score_1, score_2)
    if best_score >= 80:
        fuzzy_dob_rows.append({
            **{k: v for k, v in row.items() if k not in ['join_key_tm', 'join_key_full']},
            'join_key':     row['join_key_fbref'],
            'match_method': f'dob+fuzzy({best_score})'
        })

merge_4 = pd.DataFrame(fuzzy_dob_rows)
if not merge_4.empty:
    merge_4 = merge_4.drop_duplicates(subset='join_key')
    results.append(merge_4)
    remaining = remove_matched(remaining, set(merge_4['join_key']))
print(f"[4] DOB + fuzzy nom (TM)    : {len(merge_4):>5} | restants : {len(remaining)}")

# ══════════════════════════════════════════════════════════════════════════════
# Assemblage + fusion finale avec TM
# ══════════════════════════════════════════════════════════════════════════════
cols_base = [c for c in results[0].columns if c in df_fbref.columns or c in ['tm_id', 'match_method']]

df_with_id = pd.concat(
    [r[[c for c in cols_base if c in r.columns]] for r in results],
    ignore_index=True
)

# Fix 1 : renommer les clés TM avant le merge pour éviter la collision sur join_key
df_tm_final = df_tm.rename(columns={
    'join_key':      'tm_join_key',
    'join_key_full': 'tm_join_key_full',
    'dob_key':       'tm_dob_key'
})

df_final = pd.merge(df_with_id, df_tm_final, left_on='tm_id', right_on='player_id', how='inner')

# ── Rapport ───────────────────────────────────────────────────────────────────
print(f"\nFusion terminée          : {len(df_final)} joueurs")

print(f"\nRépartition par méthode :")
print(df_final['match_method'].value_counts().to_string())

# Fix 2 : still_missing fonctionne car join_key n'est plus écrasé
still_missing = df_fbref[~df_fbref['join_key'].isin(df_final['join_key'])]
if not still_missing.empty:
    print(f"\n{len(still_missing)} joueurs toujours non matchés :")
    print(still_missing[['Player', 'Born', 'Squad', 'join_key']].to_string(index=False))
else:
    print("\nAucun joueur manquant !")

[1] Nom exact (mapping)     :  2299 | restants : 557
[2] Fuzzy nom (mapping)     :    19 | restants : 538
[3] Nom exact + DOB (TM)    :     0 | restants : 538
[4] DOB + fuzzy nom (TM)    :     0 | restants : 538

Fusion terminée          : 2296 joueurs

Répartition par méthode :
match_method
exact_name_mapping               2278
fuzzy_name(96.55172413793103)       3
fuzzy_name(92.3076923076923)        2
fuzzy_name(96.0)                    2
fuzzy_name(96.2962962962963)        2
fuzzy_name(92.85714285714286)       2
fuzzy_name(95.65217391304348)       1
fuzzy_name(95.23809523809523)       1
fuzzy_name(93.75)                   1
fuzzy_name(97.14285714285714)       1
fuzzy_name(100.0)                   1
fuzzy_name(90.0)                    1
fuzzy_name(95.0)                    1

542 joueurs toujours non matchés :
                     Player   Born               Squad                            join_key
                Zach Abbott 2006.0   Nottingham Forest                         zach ab

In [ ]:
df_final